# Train, calibrate and evaluate the Deepfake Forensic Intelligence Suite
Run on a **GPU runtime** (Runtime → Change runtime type → T4/L4/A100).
Nothing here is pre-trained: the numbers you get are the numbers you measured.
Use three **disjoint** dataset folders, each with `real/` and `fake/`:
`train` (fits probe + spectral head), `calib` (calibration + out-of-domain model), `test` (reported results; ideally a different dataset/generator).

In [ ]:
!git clone https://github.com/mayurOG/DeepFake-Forensic-Intellegence-SUITE.git repo
%cd repo
!pip install -q -r requirements.txt
!pip install -q --no-deps facenet-pytorch
!nvidia-smi -L

Mount Drive (or download datasets here) and point the paths below at your folders.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
TRAIN = ['/content/drive/MyDrive/deepfake/train']
CALIB = ['/content/drive/MyDrive/deepfake/calib']
TEST  = ['/content/drive/MyDrive/deepfake/test']
ENCODER = 'facebook/dinov2-base'   # or a SigLIP2 / CLIP / DINOv3 checkpoint you have access to
OUT = '/content/drive/MyDrive/deepfake/artifacts'

In [ ]:
# optional: a Hugging Face token for gated models
# import os; os.environ['HF_TOKEN'] = '...'
!python -m train.run_all --train {' '.join(TRAIN)} --calib {' '.join(CALIB)} --test {' '.join(TEST)} \
    --encoder {ENCODER} --out {OUT} --mode balanced
# add --finetune for LayerNorm-only tuning, --max-items 2000 for a quick first run

In [ ]:
import json
print(json.dumps(json.load(open(f'{OUT}/run_summary.json')), indent=2)[:4000])
print(json.dumps(json.load(open(f'{OUT}/evaluation.json')), indent=2))

Read `evaluation.json` per dataset: `backbone` is the raw Tier-1 score, `cascade` is the full system, `decisions` shows the abstain rate and the false-alarm / miss rates at the chosen mode. A large gap between your train/calib results and a *different* test dataset is normal; report the cross-dataset number.

In [ ]:
# Copy artifacts back into the project to use them in the app
!cp -r {OUT} ./artifacts
!EMBED_MODEL_ID={ENCODER} python cli_scan.py -i some_image.jpg -o reports --mode balanced

## Extra evidence for the write-up
These produce the numbers a reviewer will ask for. Run after training; use a **different generator/dataset** for TEST.

In [ ]:
CT = ' '.join(f'--calib {x}' for x in CALIB) + ' ' + ' '.join(f'--test {x}' for x in TEST)
# 1) Ablation: is the adaptive cascade better than fixed fusion at matched cost?
!python -m train.evaluate_ablation {CT} --out {OUT}/ablation.json
!python -m train.evaluate_adaptive {CT} --out {OUT}/adaptive.json

In [ ]:
CT = ' '.join(f'--calib {x}' for x in CALIB) + ' ' + ' '.join(f'--test {x}' for x in TEST)
# 2) Robustness to re-encoding and black-box evasion (lower bound on fragility)
!python -m train.evaluate_robustness {CT} --out {OUT}/robustness.json
!python -m train.evaluate_adversarial --fake {TEST[0]}/fake --n 30 --out {OUT}/adversarial.json

In [ ]:
# 3) Optional: generator-family attribution (folders data/attribution/<family>/*.png)
!python -m train.train_attribution --data data/attribution --out {OUT}

In [ ]:
# 4) Audio lane with a Large SSL encoder (see Spoof-SUPERB: XLS-R / WavLM Large)
!python -m train.train_audio --train data/asvspoof --out {OUT} --encoder microsoft/wavlm-large